# Publicly Verifiable (k,n)-Threshold Quantum Secure Summation with Threshold-Paillier Input Binding

**Reference implementation and experiment suite (Google Colab).**

This notebook implements the complete protocol and regenerates every result that the paper reports.

| Phase | What happens | Implemented in |
|---|---|---|
| 0. Setup | Threshold Paillier keys (Shoup / Damgard-Jurik style, safe primes, Delta = n!) shared k-of-n | Section 2 |
| 1. Commit | Each session party publishes an additively homomorphic commitment C_j = E(m_j) with a bitwise range proof (OR-proofs of N-th residuosity) | Section 3 |
| 2. Masks | Pairwise BB84 keys give zero-sum masks r_j (sum of r_j = 0 mod d) | Section 4 |
| 3. Quantum sum | Ring QFT summation: P1 prepares QFT\|0>, each party adds (m_j + r_j) mod d as phases, the last party applies QFT^dagger and measures. Decoy qubits are checked on every hop | Section 5 |
| 4. Verify | Randomised homomorphic equality test C* = (prod C_j * E(-S_q))^rho, threshold-decrypted by any k of n key holders with zero-knowledge share proofs; accept iff the plaintext is 0 | Section 6 |

**Exact guarantee (use this wording in the paper).** An accepted output always equals the sum of the committed, range-proven inputs of the session. A participant who uses a different value in the quantum phase is detected, unless colluders cancel each other's changes exactly; in that case the accepted output is *still* the committed sum (check C10).

**What runs where.** Everything runs on Qiskit locally in Colab (the Aer simulator plus IBM fake-device noise models). Real IBM hardware is optional (E8). Run `E8_DRY_RUN` first; it executes the identical code on a fake backend.

**Security notes (also stated in the paper).**
* Input privacy against the published commitments is computational (DCR assumption), not information-theoretic.
* The adversary may corrupt at most k-1 of the n key holders. Honest key holders only produce decryption shares for the public ciphertext C*.
* The 512-bit keys used in the attack experiments are for speed only. The timing benchmark (E5) also covers 1024/2048-bit keys.
* The BB84 key establishment and decoy checks are simulated (decoys with Qiskit's stabilizer simulator); no physical channel is involved.

**How to run:** Runtime -> Run all (about 5 minutes). The conceptual-verification loop (Section 7c) must print `VERDICT` before any experiment runs. Every table is saved as CSV and every figure as PNG + PDF in `results/`. A zip of `results/` is downloaded at the end.

In [ ]:
from google.colab import userdata
token = userdata.get("IBM_TOKEN")
crn = userdata.get("IBM_INSTANCE")
print("Token loaded:", len(token), "characters")
print("CRN starts with:", crn[:4])

Token loaded: 44 characters
CRN starts with: crn:


In [ ]:
# ---- Section 0: installation (Colab) ----
import sys, subprocess
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
try:
    import qiskit, qiskit_aer, gmpy2  # noqa: F401
except ImportError:
    _pip("qiskit", "qiskit-aer", "gmpy2", "pandas", "matplotlib")
try:
    import qiskit_ibm_runtime  # noqa: F401
except ImportError:
    _pip("qiskit-ibm-runtime")   # only needed for E7 (fake backends) and E8 (hardware)

In [ ]:
SEED = 2026                 # master seed (experiments are reproducible when REPRODUCIBLE=True)
REPRODUCIBLE = True         # False -> cryptographic randomness from the OS
N_PARTIES = 5               # n : registered parties holding key shares
THRESHOLD = 3               # k : shares needed to decrypt
SESSION_SIZE = 4            # s : parties contributing inputs in one session (k <= s <= n)
INPUT_BITS = 3              # l : inputs lie in [0, 2^l)
KEY_BITS_EXPERIMENTS = 512  # Paillier modulus size used in E1-E4 (speed only)
KEY_BITS_BENCH = [512, 1024, 2048]   # sizes timed in E5
BENCH_REPS = 5              # repetitions per timed operation in E5
DECOYS_PER_HOP = 8          # D : decoy qubits inserted on every quantum hop
DECOY_ERR_TOL = 0.0         # tolerated decoy error rate (0 for noiseless runs)
TRIALS = 100                # trials per attack / configuration
SHOTS = 15                  # R : repetitions of the quantum phase (majority vote)
CHALLENGE_BITS = 128        # Sigma-protocol challenge length
STAT_SEC = 128              # statistical hiding parameter for share proofs
RUN_FAKE_BACKEND = True     # E7: IBM device noise via qiskit-ibm-runtime fake backends
RUN_ON_IBM_HARDWARE = True  # E8: real hardware (needs an IBM Quantum account)
E8_DRY_RUN = True           # ignored when RUN_ON_IBM_HARDWARE is True
HW_TRIALS = 4               # E8 sessions per (t, attack) case
VERIFY_ROUNDS = 3           # conceptual-verification loop: consecutive clean rounds required
VERIFY_MAX_ROUNDS = 6       # hard cap so the loop can never hang

# IBM credentials, read from Colab secrets (never type the key here)
from google.colab import userdata
IBM_TOKEN = userdata.get("IBM_TOKEN")        # API key
IBM_INSTANCE = userdata.get("IBM_INSTANCE")  # instance CRN

OUT = "results"

assert 2 <= THRESHOLD <= SESSION_SIZE <= N_PARTIES

TimeoutException: Requesting secret IBM_INSTANCE timed out. Secrets can only be fetched when running from the Colab UI.

In [ ]:
# ---- Section 1b: utilities ----
import os, math, time, json, hashlib, random, secrets, itertools, zipfile
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import gmpy2

os.makedirs(OUT, exist_ok=True)
RNG = random.Random(SEED) if REPRODUCIBLE else random.SystemRandom()
NP_RNG = np.random.default_rng(SEED if REPRODUCIBLE else None)

matplotlib.rcParams.update({
    "font.family": "serif", "font.size": 8, "axes.labelsize": 8, "legend.fontsize": 7,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "figure.dpi": 120, "savefig.dpi": 300,
    "axes.grid": True, "grid.alpha": 0.3,
})
IEEE_COL = 3.5  # inches, single IEEE column

def savefig(fig, name):
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(OUT, f"{name}.{ext}"), bbox_inches="tight")
    plt.show()

def savetable(df, name):
    df.to_csv(os.path.join(OUT, f"{name}.csv"), index=False)
    return df

def randbelow(n):
    return RNG.randrange(n)

def randbits(b):
    return RNG.getrandbits(b)

def powmod(b, e, m):
    if e < 0:
        b, e = invmod(b, m), -e
    return int(gmpy2.powmod(b, e, m))

def invmod(a, m):
    return int(gmpy2.invert(a, m))

def H(*parts, bits=CHALLENGE_BITS):
    """Random-oracle hash (SHA-256) used for Fiat-Shamir challenges and commit-reveal."""
    h = hashlib.sha256()
    for p in parts:
        h.update(repr(p).encode())
        h.update(b"|")
    return int.from_bytes(h.digest(), "big") % (1 << bits)

def _small_primes(limit):
    s = np.ones(limit + 1, bool)
    s[:2] = False
    for i in range(2, int(limit ** 0.5) + 1):
        if s[i]:
            s[i * i::i] = False
    return [int(p) for p in np.nonzero(s)[0] if p > 2]

_SP = _small_primes(1 << 16)
_SP_INV = [(sp, pow(2, -1, sp), pow(4, -1, sp)) for sp in _SP]

def gen_safe_prime(bits):
    """Return (p, p') with p = 2p' + 1, both prime, p of exactly `bits` bits (sieved search)."""
    qbits = bits - 1
    while True:
        q0 = randbits(qbits) | (3 << (qbits - 2)) | 1
        W = 40 * bits
        alive = np.ones(W, bool)
        for sp, inv2, inv4 in _SP_INV:
            alive[(-q0 * inv2) % sp::sp] = False          # sp | q
            alive[(-(2 * q0 + 1) * inv4) % sp::sp] = False  # sp | 2q+1
        for i in np.nonzero(alive)[0]:
            q = q0 + 2 * int(i)
            if gmpy2.is_prime(q, 1) and gmpy2.is_prime(2 * q + 1, 40) and gmpy2.is_prime(q, 40):
                return 2 * q + 1, q

print("Utilities ready.")

## 2. Threshold Paillier (k-of-n decryption with zero-knowledge share proofs)
* N = pq with safe primes p = 2p'+1 and q = 2q'+1, m = p'q', g = N+1.
* The secret exponent d satisfies d = 0 (mod m) and d = 1 (mod N). It is Shamir-shared over Z_{Nm}: s_i = f(i).
* Decryption share: c_i = c^{2 Delta s_i} mod N^2, where Delta = n!.
* Share-correctness proof: log_{c^4}(c_i^2) = log_v(v_i) (Chaum-Pedersen style, Fiat-Shamir).
* Combination: c' = prod c_i^{2 lambda_i}, where lambda_i = Delta * Lagrange coefficient (an integer). Then M = L(c') * (4 Delta^2)^{-1} mod N.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class PublicKey:
    N: int
    N2: int
    g: int
    v: int
    vk: dict
    n: int
    k: int
    Delta: int

@dataclass
class KeySet:
    pk: PublicKey
    shares: dict = field(repr=False)

def rand_unit(N):
    while True:
        r = randbelow(N - 1) + 1
        if math.gcd(r, N) == 1:
            return r

def tp_keygen(bits, n, k):
    half = bits // 2
    while True:
        p, p1 = gen_safe_prime(half)
        q, q1 = gen_safe_prime(half)
        N = p * q
        if p != q and math.gcd(p1 * q1, N) == 1 and N.bit_length() == bits:
            break
    m = p1 * q1
    d = m * invmod(m, N)                  # d = 0 mod m, d = 1 mod N
    Nm = N * m
    coeffs = [d] + [randbelow(Nm) for _ in range(k - 1)]
    shares = {i: sum(a * pow(i, e) for e, a in enumerate(coeffs)) % Nm for i in range(1, n + 1)}
    Delta = math.factorial(n)
    N2 = N * N
    while True:
        x = randbelow(N2 - 2) + 2
        if math.gcd(x, N) == 1:
            break
    v = powmod(x, 2, N2)                  # generator of the squares (w.h.p.)
    vk = {i: powmod(v, Delta * s, N2) for i, s in shares.items()}
    return KeySet(PublicKey(N, N2, N + 1, v, vk, n, k, Delta), shares)

def encrypt(pk, msg, r=None):
    r = rand_unit(pk.N) if r is None else r
    return (1 + (msg % pk.N) * pk.N) * powmod(r, pk.N, pk.N2) % pk.N2

def ct_add(pk, *cts):
    out = 1
    for c in cts:
        out = out * c % pk.N2
    return out

def decryption_share(keys, i, c):
    pk = keys.pk
    x = pk.Delta * keys.shares[i]
    ci = powmod(c, 2 * x, pk.N2)
    ct = powmod(c, 4, pk.N2)
    ci2 = powmod(ci, 2, pk.N2)
    rbits = x.bit_length() + CHALLENGE_BITS + STAT_SEC
    r = randbits(rbits)
    a = powmod(ct, r, pk.N2)
    b = powmod(pk.v, r, pk.N2)
    e = H("share", pk.N, pk.v, ct, pk.vk[i], ci2, a, b)
    z = r + e * x
    return ci, (a, b, z)

def verify_share(pk, i, c, ci, proof):
    a, b, z = proof
    ct = powmod(c, 4, pk.N2)
    ci2 = powmod(ci, 2, pk.N2)
    e = H("share", pk.N, pk.v, ct, pk.vk[i], ci2, a, b)
    return (powmod(ct, z, pk.N2) == a * powmod(ci2, e, pk.N2) % pk.N2 and
            powmod(pk.v, z, pk.N2) == b * powmod(pk.vk[i], e, pk.N2) % pk.N2)

def lagrange_int(S, j, Delta):
    num, den = Delta, 1
    for jp in S:
        if jp != j:
            num *= -jp
            den *= (j - jp)
    assert num % den == 0
    return num // den

def combine_shares(pk, share_dict):
    S = sorted(share_dict)
    assert len(S) >= pk.k
    S = S[:pk.k]
    cp = 1
    for j in S:
        cp = cp * powmod(share_dict[j], 2 * lagrange_int(S, j, pk.Delta), pk.N2) % pk.N2
    L = (cp - 1) // pk.N
    return L * invmod(4 * pk.Delta * pk.Delta % pk.N, pk.N) % pk.N

print("Threshold Paillier ready.")

## 3. Input commitments with bitwise range proofs
Each bit b is encrypted as c = g^b r^N. A 1-out-of-2 OR-proof (Cramer-Damgard-Schoenmakers composition of the N-th residuosity Sigma-protocol) shows that c or c*g^{-1} is an N-th residue, i.e. that b is 0 or 1.
The commitment is C = prod c_j^{2^j}, so it provably encrypts a value in [0, 2^l).

In [ ]:
def _or_prove(pk, c, b, r, ctx):
    N, N2, T = pk.N, pk.N2, 1 << CHALLENGE_BITS
    u = [c, c * invmod(pk.g, N2) % N2]
    s = 1 - b
    e, a, z = [0, 0], [0, 0], [0, 0]
    e[s] = randbelow(T)
    z[s] = rand_unit(N)
    a[s] = powmod(z[s], N, N2) * powmod(u[s], -e[s], N2) % N2
    rho = rand_unit(N)
    a[b] = powmod(rho, N, N2)
    E = H("or", N, c, a[0], a[1], ctx)
    e[b] = (E - e[s]) % T
    z[b] = rho * powmod(r, e[b], N) % N
    return (a[0], a[1], e[0], e[1], z[0], z[1])

def _or_verify(pk, c, proof, ctx):
    N, N2, T = pk.N, pk.N2, 1 << CHALLENGE_BITS
    a0, a1, e0, e1, z0, z1 = proof
    if (e0 + e1) % T != H("or", N, c, a0, a1, ctx):
        return False
    u1 = c * invmod(pk.g, N2) % N2
    return (powmod(z0, N, N2) == a0 * powmod(c, e0, N2) % N2 and
            powmod(z1, N, N2) == a1 * powmod(u1, e1, N2) % N2)

def commit_input(pk, value, ell, pid, sid, cheat_out_of_range=False):
    """Returns (C, bit_ciphertexts, proofs). If cheat_out_of_range, bit 0 secretly encrypts 2."""
    cts, proofs = [], []
    for j in range(ell):
        b = (value >> j) & 1
        r = rand_unit(pk.N)
        enc_val = 2 if (cheat_out_of_range and j == 0) else b
        c = encrypt(pk, enc_val, r)
        proofs.append(_or_prove(pk, c, b if not (cheat_out_of_range and j == 0) else 1, r, (sid, pid, j)))
        cts.append(c)
    C = 1
    for j, c in enumerate(cts):
        C = C * powmod(c, 1 << j, pk.N2) % pk.N2
    return C, cts, proofs

def verify_commitment(pk, C, cts, proofs, pid, sid):
    if not all(_or_verify(pk, c, pr, (sid, pid, j)) for j, (c, pr) in enumerate(zip(cts, proofs))):
        return False
    Cr = 1
    for j, c in enumerate(cts):
        Cr = Cr * powmod(c, 1 << j, pk.N2) % pk.N2
    return Cr == C

print("Commitments ready.")

## 4. BB84 pairwise keys and zero-sum masks
BB84 is simulated statistically: random bits and bases, optional intercept-resend eavesdropper, optional channel flips, sifting, and a QBER test on a sample (abort above 11%).
The mask for party j is r_j = sum_{l>j} K_{jl} - sum_{l<j} K_{lj} (mod d), so that sum_j r_j = 0 (mod d).

In [ ]:
def bb84(n_raw, eve_prob=0.0, flip_prob=0.0, sample_frac=0.25, qber_max=0.11):
    a_bits = NP_RNG.integers(0, 2, n_raw)
    a_bas = NP_RNG.integers(0, 2, n_raw)
    b_bas = NP_RNG.integers(0, 2, n_raw)
    sent_bits, sent_bas = a_bits.copy(), a_bas.copy()
    eve = NP_RNG.random(n_raw) < eve_prob
    e_bas = NP_RNG.integers(0, 2, n_raw)
    e_bits = np.where(e_bas == a_bas, a_bits, NP_RNG.integers(0, 2, n_raw))
    sent_bits = np.where(eve, e_bits, sent_bits)
    sent_bas = np.where(eve, e_bas, sent_bas)
    b_bits = np.where(b_bas == sent_bas, sent_bits, NP_RNG.integers(0, 2, n_raw))
    b_bits = b_bits ^ (NP_RNG.random(n_raw) < flip_prob)
    keep = a_bas == b_bas
    ka, kb = a_bits[keep], b_bits[keep]
    ns = max(1, int(len(ka) * sample_frac))
    idx = NP_RNG.permutation(len(ka))
    samp, rest = idx[:ns], idx[ns:]
    qber = float(np.mean(ka[samp] != kb[samp]))
    return {"key_a": ka[rest], "key_b": kb[rest], "qber": qber, "abort": qber > qber_max}

def bits_to_int(bits):
    return int("".join(str(int(b)) for b in bits), 2) if len(bits) else 0

def zero_sum_masks(parties, t, eve_pair=None, eve_prob=0.0, n_raw=None):
    """Returns (masks dict, aborted_pair or None, qbers).
    Each party uses its OWN copy of every pairwise key, so an undetected eavesdropper who
    corrupts a key breaks the zero-sum property (and the equality test then rejects)."""
    d = 1 << t
    Ka, Kb, qbers = {}, {}, {}
    n_raw = n_raw or (512 + 16 * t)
    for a, b in itertools.combinations(parties, 2):
        while True:
            res = bb84(n_raw, eve_prob=eve_prob if (a, b) == eve_pair else 0.0)
            if len(res["key_a"]) >= t:
                break
        qbers[(a, b)] = res["qber"]
        if res["abort"]:
            return None, (a, b), qbers
        Ka[(a, b)] = bits_to_int(res["key_a"][:t]) % d   # held by the lower-index party a
        Kb[(a, b)] = bits_to_int(res["key_b"][:t]) % d   # held by the higher-index party b
    masks = {}
    for j in parties:
        masks[j] = (sum(Ka[(j, l)] for l in parties if l > j) - sum(Kb[(l, j)] for l in parties if l < j)) % d
    return masks, None, qbers

print("BB84 / masks ready.")

## 5. Quantum ring summation (QFT phase adder) and decoy checks
* Qubit i of the t-qubit register carries weight 2^i (Qiskit little-endian); d = 2^t.
* P1 prepares QFT|0>. Party j adds a_j = (m_j + r_j) mod d with phase gates P(2 pi a_j 2^i / d). The last party applies QFT^dagger and measures, obtaining sum a_j = sum m_j (mod d).
* The register width is chosen so the sum never wraps: d >= s(2^l - 1) + 1.
* The quantum phase is repeated R times; S_q is the unique most frequent outcome (plurality). On a tie the session aborts, so ties are never broken arbitrarily.
* Every hop carries D decoy qubits drawn at random from {|0>, |1>, |+>, |->}. The receiver measures them in the announced basis.
* Noise model (IBM-like): depolarizing error on sx/x (p1) and cx (p2), readout error, and per-hop channel depolarizing error on id gates.

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

def qft_circ(t):
    qc = QuantumCircuit(t, name="QFT")
    for j in reversed(range(t)):
        qc.h(j)
        for l in reversed(range(j)):
            qc.cp(math.pi / 2 ** (j - l), l, j)
    for i in range(t // 2):
        qc.swap(i, t - 1 - i)
    return qc

def add_phase(qc, qreg, a, d):
    for i in range(len(qreg)):
        qc.p(2 * math.pi * (a % d) * (2 ** i) / d, qreg[i])

def register_width(s, ell):
    return max(1, math.ceil(math.log2(s * ((1 << ell) - 1) + 1)))

def build_session_circuit(t, contributions, hop_attacks=None, channel=False):
    """contributions: values added by the parties in ring order.
    hop_attacks: {hop: ("ir",) | ("phase", delta)}; hop h = transmission from party h to party h+1."""
    hop_attacks = hop_attacks or {}
    d = 1 << t
    q = QuantumRegister(t, "q")
    out = ClassicalRegister(t, "out")
    qc = QuantumCircuit(q, out)
    qc.compose(qft_circ(t), q, inplace=True)
    for idx, a in enumerate(contributions):
        if idx > 0:
            hop = idx - 1
            qc.barrier(label=f"hop{hop}")   # party boundary: no gate merging across parties
            if channel:
                for qq in q:
                    qc.id(qq)
            atk = hop_attacks.get(hop)
            if atk and atk[0] == "ir":
                ev = ClassicalRegister(t, f"eve{hop}")
                qc.add_register(ev)
                for i, qq in enumerate(q):
                    if RNG.random() < 0.5:     # X-basis measure-and-resend
                        qc.h(qq); qc.measure(qq, ev[i]); qc.h(qq)
                    else:                      # Z-basis measure-and-resend
                        qc.measure(qq, ev[i])
            elif atk and atk[0] == "phase":
                add_phase(qc, q, atk[1], d)
        add_phase(qc, q, a, d)
    qc.compose(qft_circ(t).inverse(), q, inplace=True)
    qc.measure(q, out)
    return qc

def make_noise(p1=0.0, p2=0.0, pro=0.0, pch=0.0):
    if max(p1, p2, pro, pch) == 0:
        return None
    nm = NoiseModel(basis_gates=["id", "rz", "sx", "x", "cx"])
    if p1 > 0:
        nm.add_all_qubit_quantum_error(depolarizing_error(p1, 1), ["sx", "x"])
    if p2 > 0:
        nm.add_all_qubit_quantum_error(depolarizing_error(p2, 2), ["cx"])
    if pch > 0:
        nm.add_all_qubit_quantum_error(depolarizing_error(pch, 1), ["id"])
    if pro > 0:
        nm.add_all_qubit_readout_error(ReadoutError([[1 - pro, pro], [pro, 1 - pro]]))
    return nm

_SIM_CACHE = {}
def get_sim(noise, method="automatic"):
    seed = SEED if REPRODUCIBLE else None
    if noise is not None:                      # noisy simulators are built fresh (no stale cache)
        return AerSimulator(noise_model=noise, method=method, seed_simulator=seed)
    if method not in _SIM_CACHE:
        _SIM_CACHE[method] = AerSimulator(method=method, seed_simulator=seed)
    return _SIM_CACHE[method]

def run_circuits(circuits, shots, noise=None):
    """Runs a batch of circuits; returns a list of Counters over the 'out' register value."""
    sim = get_sim(noise)
    tq = transpile(circuits, sim, optimization_level=0, seed_transpiler=SEED)
    res = sim.run(tq, shots=shots).result()
    outs = []
    for i in range(len(circuits)):
        c = Counter()
        for key, v in res.get_counts(i).items():
            c[int(key.split()[-1], 2)] += v
        outs.append(c)
    return outs

def plurality(counts):
    """Unique most frequent outcome, or None on a tie (the protocol then aborts).
    A deterministic, conservative rule: ties are never resolved arbitrarily."""
    top = counts.most_common(2)
    if not top or (len(top) == 2 and top[0][1] == top[1][1]):
        return None
    return top[0][0]

def wilson(k, n, z=1.96):
    """95% Wilson score interval for a binomial proportion (returned in the same units as k/n)."""
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (max(0.0, c - h), min(1.0, c + h))

def decoy_hop_circuit(D, attacked, pch=0.0):
    """One hop of D decoys. Returns (circuit, prepared bits, bases). Channel noise = random Paulis."""
    q = QuantumRegister(D, "d")
    out = ClassicalRegister(D, "out")
    qc = QuantumCircuit(q, out)
    bits = [RNG.getrandbits(1) for _ in range(D)]
    bases = [RNG.getrandbits(1) for _ in range(D)]      # 0 = Z, 1 = X
    for i in range(D):
        if bits[i]: qc.x(q[i])
        if bases[i]: qc.h(q[i])
    if pch > 0:
        for i in range(D):
            if RNG.random() < 3 * pch / 4:
                getattr(qc, RNG.choice(["x", "y", "z"]))(q[i])
    if attacked:
        ev = ClassicalRegister(D, "eve")
        qc.add_register(ev)
        for i in range(D):
            if RNG.getrandbits(1):
                qc.h(q[i]); qc.measure(q[i], ev[i]); qc.h(q[i])
            else:
                qc.measure(q[i], ev[i])
    for i in range(D):
        if bases[i]: qc.h(q[i])
    qc.measure(q, out)
    return qc, bits, bases

def run_decoy_checks(hops_attacked, D, pch=0.0):
    """hops_attacked: list of bools, one per hop. Returns list of error rates per hop."""
    if D == 0:
        return [0.0] * len(hops_attacked)
    circs, preps = [], []
    for att in hops_attacked:
        qc, bits, _ = decoy_hop_circuit(D, att, pch)
        circs.append(qc); preps.append(bits)
    sim = get_sim(None, "stabilizer")
    res = sim.run(circs, shots=1).result()
    rates = []
    for i, bits in enumerate(preps):
        key = list(res.get_counts(i).keys())[0].split()[-1]
        meas = [int(ch) for ch in reversed(key)]
        rates.append(sum(m != b for m, b in zip(meas, bits)) / D)
    return rates

print("Quantum layer ready.")

## 6. Full protocol
`run_protocol` executes all phases and returns a transcript with the outcome, the layer that detected any attack, and the identified party (if any).

Supported attacks: `none`, `input_substitution`, `phase_tamper` (an adaptive channel attacker who knows the decoy positions), `intercept_resend`, `forge_result`, `bad_share`, `out_of_range`, `bb84_eavesdrop`, and `compensating_collusion` (two colluders shift their inputs by +delta and -delta; used in check C10 to demonstrate the exact guarantee).

In [ ]:
ATTACKS = ["none", "input_substitution", "phase_tamper", "intercept_resend",
           "forge_result", "bad_share", "out_of_range", "bb84_eavesdrop"]

def joint_random(pk, parties, sid):
    """Commit-reveal coin tossing for rho in Z_N^*."""
    while True:
        picks = {j: randbelow(pk.N) for j in parties}
        commits = {j: H("rho", sid, j, picks[j], bits=256) for j in parties}
        assert all(commits[j] == H("rho", sid, j, picks[j], bits=256) for j in parties)
        rho = sum(picks.values()) % pk.N
        if rho and math.gcd(rho, pk.N) == 1:
            return rho

def equality_test(keys, commits, S_q, session, decryptor_pool, sid, bad_share_party=None, log=None):
    """Phase 4. Publicly recomputable C* = (prod C_j * E(-S_q; 1))^rho, threshold-decrypted by the first
    k decryptors whose share proofs verify. Returns accept iff the plaintext of C* is 0.
    Only 0 / non-zero is learned: for a mismatch delta, the plaintext is rho*delta mod N (uniformly random)."""
    log = log or (lambda *a: None)
    pk = keys.pk
    C_sum = ct_add(pk, *[commits[j] for j in session])
    C_diff = C_sum * (1 + (pk.N - (S_q % pk.N)) * pk.N) % pk.N2      # C_sum * E(-S_q; r=1)
    rho = joint_random(pk, session, sid)
    C_star = powmod(C_diff, rho, pk.N2)
    public_C_star = C_star                                             # anyone can recompute this
    shares, cheaters = {}, []
    pool = list(decryptor_pool)
    if bad_share_party is not None:
        pool = [bad_share_party] + [j for j in pool if j != bad_share_party]
    for j in pool:
        assert C_star == public_C_star                                 # honest-share policy
        cj, proof = decryption_share(keys, j, C_star)
        if j == bad_share_party:
            cj = cj * powmod(randbelow(pk.N2 - 2) + 2, 2, pk.N2) % pk.N2
        if verify_share(pk, j, C_star, cj, proof):
            shares[j] = cj
        else:
            cheaters.append(j)
            log(f"[decrypt] share of party {j} INVALID -> party {j} identified and excluded")
        if len(shares) == pk.k:
            break
    if len(shares) < pk.k:
        return {"status": "abort", "accepted": False, "cheaters": cheaters, "decryptors": sorted(shares)}
    M = combine_shares(pk, shares)
    log(f"[verify] decryptors {sorted(shares)}; plaintext of C* is {'0' if M == 0 else 'nonzero (random)'} -> "
        f"{'ACCEPT' if M == 0 else 'REJECT'}")
    return {"status": "done", "accepted": M == 0, "cheaters": cheaters, "decryptors": sorted(shares),
            "plaintext": M}

def run_protocol(keys, inputs, session, decryptor_pool, ell, attack="none",
                 noise=None, pch=0.0, shots=SHOTS, D=DECOYS_PER_HOP, decoy_tol=None, sid=None, verbose=False):
    pk = keys.pk
    s = len(session)
    t = register_width(s, ell)
    d = 1 << t
    sid = sid if sid is not None else randbits(64)
    adv = session[1]                       # adversarial session party (when relevant)
    tr = {"attack": attack, "t": t, "d": d, "true_sum": sum(inputs[j] for j in session),
          "accepted": False, "detected_by": None, "identified": None, "S_q": None, "status": ""}
    log = (lambda *a: print(*a)) if verbose else (lambda *a: None)

    # Phase 1: commitments with range proofs
    commits = {}
    for j in session:
        C, cts, prf = commit_input(pk, inputs[j], ell, j, sid,
                                   cheat_out_of_range=(attack == "out_of_range" and j == adv))
        if not verify_commitment(pk, C, cts, prf, j, sid):
            tr.update(status="abort: invalid range proof", detected_by="range_proof", identified=j)
            log(f"[commit] party {j}: range proof INVALID -> abort, party {j} identified")
            return tr
        commits[j] = C
        log(f"[commit] party {j}: C_{j} = {str(C)[:20]}... range proof valid")

    # Phase 2: BB84 pairwise keys and zero-sum masks
    eve_pair = (session[0], session[1]) if attack == "bb84_eavesdrop" else None
    masks, bad_pair, qbers = zero_sum_masks(session, t, eve_pair, eve_prob=1.0)
    if masks is None:
        tr.update(status=f"abort: QBER {qbers[bad_pair]:.2f} on BB84 link {bad_pair}",
                  detected_by="bb84_qber", identified=bad_pair)
        log(f"[bb84] link {bad_pair}: QBER={qbers[bad_pair]:.2f} > 0.11 -> abort")
        return tr
    log(f"[masks] r = {masks} (sum mod {d} = {sum(masks.values()) % d})")

    # Phase 3: quantum ring summation
    contrib = []
    comp_delta = (1 + randbelow(d - 1)) if attack == "compensating_collusion" else 0
    for j in session:
        m = inputs[j]
        if attack == "input_substitution" and j == adv:
            m = (m + 1 + randbelow(d - 1)) % d      # uses m' != m in the quantum phase
        if attack == "compensating_collusion" and j in (session[1], session[2]):
            m = (m + comp_delta) % d if j == session[1] else (m - comp_delta) % d
        contrib.append((m + masks[j]) % d)
    hop_attacks, hops_attacked = {}, [False] * (s - 1)
    atk_hop = randbelow(s - 1)
    if attack == "intercept_resend":
        hop_attacks[atk_hop] = ("ir",)
        hops_attacked[atk_hop] = True
    if attack == "phase_tamper":
        hop_attacks[atk_hop] = ("phase", 1 + randbelow(d - 1))   # adaptive: decoys untouched
    rates = run_decoy_checks(hops_attacked, D, pch)
    tol = DECOY_ERR_TOL if decoy_tol is None else decoy_tol
    bad_hops = [h for h, r in enumerate(rates) if r > tol]
    tr["decoy_error_rates"] = rates
    if bad_hops:
        tr.update(status=f"abort: decoy errors on hop(s) {bad_hops}", detected_by="decoy",
                  identified=("channel", bad_hops))
        log(f"[decoy] error rates {rates} -> abort on hop(s) {bad_hops}")
        return tr
    qc = build_session_circuit(t, contrib, hop_attacks, channel=pch > 0)
    counts = run_circuits([qc], shots, noise)[0]
    S_q = plurality(counts)
    tr["p_correct_shot"] = counts.get(tr["true_sum"], 0) / shots
    if S_q is None:
        tr.update(status="abort: no unique plurality outcome", detected_by="plurality_tie")
        log(f"[quantum] tie among top outcomes {counts.most_common(3)} -> abort")
        return tr
    if attack == "forge_result":
        S_q = (S_q + 1 + randbelow(d - 1)) % d
    tr["S_q"] = S_q
    log(f"[quantum] t={t}, d={d}, contributions={contrib}, counts(top3)={counts.most_common(3)}, S_q={S_q}")

    # Phase 4: randomised homomorphic equality test with threshold decryption
    bad = adv if attack == "bad_share" else None
    ver = equality_test(keys, commits, S_q, session, decryptor_pool, sid, bad_share_party=bad, log=log)
    tr["decryptors"] = ver["decryptors"]
    if ver["status"] == "abort":
        tr.update(status="abort: not enough valid shares", detected_by="share_proof", identified=ver["cheaters"])
        return tr
    tr["accepted"] = ver["accepted"]
    if ver["cheaters"]:
        tr["identified"] = ver["cheaters"][0]
        tr["detected_by"] = "share_proof"
    if not tr["accepted"]:
        tr["detected_by"] = "equality_test"
    tr["status"] = "ACCEPT" if tr["accepted"] else "REJECT (equality test)"
    return tr

print("Protocol ready.")

## 7. Sanity tests (all must pass)

In [ ]:
from qiskit.quantum_info import Operator

t0 = time.time()
KEYS = tp_keygen(KEY_BITS_EXPERIMENTS, N_PARTIES, THRESHOLD)
PK = KEYS.pk
print(f"Key generation ({KEY_BITS_EXPERIMENTS}-bit N): {time.time() - t0:.2f} s")

# T1 threshold decryption with every k-subset
for S in itertools.combinations(range(1, N_PARTIES + 1), THRESHOLD):
    msg = randbelow(10 ** 6)
    c = encrypt(PK, msg)
    sh = {}
    for j in S:
        cj, pr = decryption_share(KEYS, j, c)
        assert verify_share(PK, j, c, cj, pr)
        sh[j] = cj
    assert combine_shares(PK, sh) == msg
print("T1 threshold decryption for all k-subsets: PASS")

# T2 homomorphic addition
a_, b_ = 1234, 5678
c_ab = ct_add(PK, encrypt(PK, a_), encrypt(PK, b_))
sh = {j: decryption_share(KEYS, j, c_ab)[0] for j in range(1, THRESHOLD + 1)}
assert combine_shares(PK, sh) == a_ + b_
print("T2 homomorphic addition: PASS")

# T3 k-1 shares do not decrypt (privacy sanity check)
c = encrypt(PK, 42)
sh = {j: decryption_share(KEYS, j, c)[0] for j in range(1, THRESHOLD)}
fake_pk = PublicKey(**{**PK.__dict__, "k": THRESHOLD - 1})
assert combine_shares(fake_pk, sh) != 42
print("T3 (k-1) shares fail to decrypt: PASS")

# T4 tampered share is rejected
cj, pr = decryption_share(KEYS, 1, c)
assert not verify_share(PK, 1, c, cj * 4 % PK.N2, pr)
print("T4 tampered decryption share rejected: PASS")

# T5 range proofs: honest accepted, out-of-range rejected
C, cts, prf = commit_input(PK, 5, INPUT_BITS, 1, 0)
assert verify_commitment(PK, C, cts, prf, 1, 0)
C, cts, prf = commit_input(PK, 5, INPUT_BITS, 1, 0, cheat_out_of_range=True)
assert not verify_commitment(PK, C, cts, prf, 1, 0)
print("T5 range proof completeness/soundness: PASS")

# T6 QFT unitary equals the textbook definition
for t in (2, 3, 4):
    Nn = 1 << t
    F = np.array([[np.exp(2j * np.pi * x * y / Nn) for x in range(Nn)] for y in range(Nn)]) / np.sqrt(Nn)
    assert np.allclose(Operator(qft_circ(t)).data, F)
print("T6 QFT unitary: PASS")

# T7 exhaustive ring-adder correctness (t = 3, three parties)
circs, exp = [], []
for vals in itertools.product(range(8), repeat=3):
    circs.append(build_session_circuit(3, list(vals)))
    exp.append(sum(vals) % 8)
outs = run_circuits(circs, 1)
assert all(list(o)[0] == e for o, e in zip(outs, exp))
print("T7 exhaustive ring adder (512 cases): PASS")

# T8 zero-sum masks
m_, _, _ = zero_sum_masks([1, 2, 3, 4], 5)
assert sum(m_.values()) % 32 == 0
print("T8 zero-sum masks: PASS")
print("All sanity tests passed.")

## 7b. Baseline: qubit emulation of Sutradhar & Om (IEEE TCAS-II 2020)
This emulates the base protocol's structure on qubits (d = 2^t):
1. P1 prepares QFT|0> and copies it with SUM gates (bitwise CNOTs), giving sum_l |l>|l>...|l>.
2. Each player applies the QFT and the generalized Pauli shift U_{A_j,0} (a Draper adder), measures, and broadcasts m_j + A_j.
3. The output is the sum of the broadcasts (mod d).

The Lagrange "shadows" A_j of the base paper form an additive sharing of the secret sum, so they are emulated here as additive shares mod 2^t (the base paper works over a prime d).
The base protocol has no verification step, so any corruption of the output is silent.

In [ ]:
def base_protocol_circuit(t, shadows, phase_attack=None):
    """phase_attack = (player_index, delta): the phase shift Z^delta applied to that player's transmitted register."""
    d = 1 << t
    k = len(shadows)
    regs = [QuantumRegister(t, f"p{j}") for j in range(k)]
    outs = [ClassicalRegister(t, f"m{j}") for j in range(k)]
    qc = QuantumCircuit(*regs, *outs)
    qc.compose(qft_circ(t), regs[0], inplace=True)          # P1: QFT|0> = sum_l |l>
    for j in range(1, k):                                   # SUM gates (copy in the computational basis)
        for i in range(t):
            qc.cx(regs[0][i], regs[j][i])
    qc.barrier(label="send")
    if phase_attack is not None:
        add_phase(qc, regs[phase_attack[0]], phase_attack[1], d)   # Z^delta on one travelling register
    for j, A in enumerate(shadows):
        qc.compose(qft_circ(t), regs[j], inplace=True)      # player's QFT
        qc.compose(qft_circ(t), regs[j], inplace=True)      # U_{A,0} = Draper adder: QFT, phase, QFT^-1
        add_phase(qc, regs[j], A, d)
        qc.compose(qft_circ(t).inverse(), regs[j], inplace=True)
        qc.measure(regs[j], outs[j])
    return qc

def base_protocol_output(counts_key, t):
    return sum(int(tok, 2) for tok in counts_key.split()) % (1 << t)

def base_protocol_trials(t, k, trials, attack="none", noise=None):
    d = 1 << t
    circs, truths, deltas = [], [], []
    for _ in range(trials):
        S = randbelow(d)
        sh = [randbelow(d) for _ in range(k - 1)]
        sh.append((S - sum(sh)) % d)
        delta = 1 + randbelow(d - 1)
        pa = (1 + randbelow(k - 1), delta) if attack == "phase" else None
        circs.append(base_protocol_circuit(t, sh, pa))
        truths.append(S)
        deltas.append(delta)
    sim = get_sim(noise)
    res = sim.run(transpile(circs, sim, optimization_level=0, seed_transpiler=SEED), shots=1).result()
    outs = []
    for i in range(trials):
        key = list(res.get_counts(i).keys())[0]
        out = base_protocol_output(key, t)
        if attack == "broadcast":                   # a dishonest player adds delta to its broadcast
            out = (out + deltas[i]) % d
        outs.append(out)
    return truths, outs, deltas

## 7c. Conceptual-verification loop
Each round re-checks every property the paper claims, using fresh randomness:
* **C1** threshold decryption is correct for every k-subset, including edge messages;
* **C2** the equality test is complete (accepts a correct sum) and sound (rejects any mismatch), and its plaintext on a mismatch is randomised;
* **C3** range proofs are complete for every input, sound, and bound to their context;
* **C4** share proofs reject tampered, replayed, and misattributed shares;
* **C5** the ring adder is exact for random widths and party counts;
* **C6** a phase shift on the channel shifts the sum by exactly delta;
* **C7** masks sum to zero and each mask is uniform;
* **C8** BB84 QBER is about 0 when honest and about 0.25 with an intercept-resend eavesdropper;
* **C9** decoy error rate is 0 when honest and about 0.25 per decoy under intercept-resend;
* **C10** end-to-end semantics, including the exact guarantee: an accepted output equals the sum of the committed, range-proven inputs;
* **C11** the baseline is correct when honest and silently corrupted by a phase shift or a false broadcast;
* **C12** determinism: identical seeds give identical results;
* **C13** the register width never lets the sum wrap around.

The loop stops only after VERIFY_ROUNDS consecutive clean rounds. If any check fails, it raises immediately with details (a deterministic bug cannot be fixed by looping), and it never runs more than VERIFY_MAX_ROUNDS rounds. The RNG state is saved and restored, so later experiments are unaffected.

In [ ]:
SESSION = list(range(1, SESSION_SIZE + 1))
POOL = list(range(N_PARTIES, 0, -1))        # decryptors: any k of n, here the last parties first

def _check(name, cond, detail=""):
    if not cond:
        raise AssertionError(f"{name} FAILED {detail}")

def conceptual_round(keys, rnd):
    pk = keys.pk
    report = {}
    # C1
    msgs = [0, 1, pk.N - 1, randbelow(pk.N)]
    for S in itertools.combinations(range(1, pk.n + 1), pk.k):
        for msg in msgs:
            c = encrypt(pk, msg)
            sh = {}
            for j in S:
                cj, pr = decryption_share(keys, j, c)
                _check("C1 share proof", verify_share(pk, j, c, cj, pr))
                sh[j] = cj
            _check("C1 decrypt", combine_shares(pk, sh) == msg, f"S={S} msg={msg}")
    report["C1"] = "ok"
    # C2
    sess = list(range(1, SESSION_SIZE + 1))
    vals = {j: randbelow(1 << INPUT_BITS) for j in sess}
    commits = {j: commit_input(pk, vals[j], INPUT_BITS, j, rnd)[0] for j in sess}
    true = sum(vals.values())
    _check("C2 completeness", equality_test(keys, commits, true, sess, POOL, rnd)["accepted"])
    plains = set()
    for delta in [1, 2, (1 << INPUT_BITS), randbelow(pk.N - 1) + 1, pk.N - 1]:
        r = equality_test(keys, commits, (true + delta) % pk.N, sess, POOL, rnd)
        _check("C2 soundness", not r["accepted"] and r["plaintext"] != 0, f"delta={delta}")
        plains.add(r["plaintext"])
    r1 = equality_test(keys, commits, true + 1, sess, POOL, (rnd, "a"))["plaintext"]
    r2 = equality_test(keys, commits, true + 1, sess, POOL, (rnd, "b"))["plaintext"]
    _check("C2 randomised plaintext", r1 != r2 and len(plains) == 5)
    report["C2"] = "ok"
    # C3
    for v in range(1 << INPUT_BITS):
        C, cts, prf = commit_input(pk, v, INPUT_BITS, 1, rnd)
        _check("C3 completeness", verify_commitment(pk, C, cts, prf, 1, rnd), f"v={v}")
        sh = {j: decryption_share(keys, j, C)[0] for j in range(1, pk.k + 1)}
        _check("C3 commitment opens to v", combine_shares(pk, sh) == v)
    C, cts, prf = commit_input(pk, 3, INPUT_BITS, 1, rnd, cheat_out_of_range=True)
    _check("C3 soundness", not verify_commitment(pk, C, cts, prf, 1, rnd))
    C, cts, prf = commit_input(pk, 3, INPUT_BITS, 1, rnd)
    _check("C3 context binding (party)", not verify_commitment(pk, C, cts, prf, 2, rnd))
    _check("C3 context binding (session)", not verify_commitment(pk, C, cts, prf, 1, (rnd, "other")))
    report["C3"] = "ok"
    # C4
    c1, c2 = encrypt(pk, 5), encrypt(pk, 6)
    cj, pr = decryption_share(keys, 1, c1)
    _check("C4 tampered share", not verify_share(pk, 1, c1, cj * 4 % pk.N2, pr))
    _check("C4 replay on other ciphertext", not verify_share(pk, 1, c2, cj, pr))
    _check("C4 misattributed share", not verify_share(pk, 2, c1, cj, pr))
    report["C4"] = "ok"
    # C5 + C6
    circs, exp = [], []
    for _ in range(40):
        t = 2 + randbelow(5); s_ = 2 + randbelow(5); d = 1 << t
        vals_q = [randbelow(d) for _ in range(s_)]
        if randbelow(2):
            delta = 1 + randbelow(d - 1)
            circs.append(build_session_circuit(t, vals_q, {randbelow(s_ - 1): ("phase", delta)}))
            exp.append((sum(vals_q) + delta) % d)
        else:
            circs.append(build_session_circuit(t, vals_q))
            exp.append(sum(vals_q) % d)
    outs = run_circuits(circs, 3)
    _check("C5/C6 ring adder & phase-shift model", all(len(o) == 1 and list(o)[0] == e for o, e in zip(outs, exp)))
    report["C5"] = report["C6"] = "ok"
    # C7
    hist = Counter()
    for _ in range(300):
        m_, _, _ = zero_sum_masks([1, 2, 3], 2)
        _check("C7 zero-sum", sum(m_.values()) % 4 == 0)
        hist[m_[1]] += 1
    _check("C7 mask uniformity", all(40 <= hist[v] <= 110 for v in range(4)), str(dict(hist)))
    report["C7"] = "ok"
    # C8
    honest_q = np.mean([bb84(2000)["qber"] for _ in range(5)])
    eve_q = np.mean([bb84(2000, eve_prob=1.0)["qber"] for _ in range(5)])
    _check("C8 BB84 QBER", honest_q == 0.0 and 0.20 <= eve_q <= 0.30, f"{honest_q}, {eve_q}")
    report["C8"] = f"QBER honest={honest_q:.3f}, Eve={eve_q:.3f}"
    # C9
    h_rates = run_decoy_checks([False] * 50, 16)
    a_rates = run_decoy_checks([True] * 200, 16)
    _check("C9 decoys", max(h_rates) == 0 and 0.22 <= float(np.mean(a_rates)) <= 0.28, f"{np.mean(a_rates)}")
    report["C9"] = f"decoy error under IR={np.mean(a_rates):.3f}"
    # C10
    inp = {j: randbelow(1 << INPUT_BITS) for j in SESSION}
    tot = sum(inp.values())
    r = run_protocol(keys, inp, SESSION, POOL, INPUT_BITS, shots=1)
    _check("C10 honest", r["accepted"] and r["S_q"] == tot)
    r = run_protocol(keys, inp, SESSION, POOL, INPUT_BITS, attack="input_substitution", shots=1)
    _check("C10 substitution rejected", not r["accepted"] and r["detected_by"] == "equality_test")
    r = run_protocol(keys, inp, SESSION, POOL, INPUT_BITS, attack="forge_result", shots=1)
    _check("C10 forgery rejected", not r["accepted"])
    r = run_protocol(keys, inp, SESSION, POOL, INPUT_BITS, attack="phase_tamper", shots=1)
    _check("C10 phase tamper rejected", not r["accepted"])
    r = run_protocol(keys, inp, SESSION, POOL, INPUT_BITS, attack="bad_share", shots=1)
    _check("C10 bad share identified, result still correct",
           r["accepted"] and r["S_q"] == tot and r["identified"] == SESSION[1])
    r = run_protocol(keys, inp, SESSION, POOL, INPUT_BITS, attack="out_of_range", shots=1)
    _check("C10 out-of-range identified", r["detected_by"] == "range_proof" and r["identified"] == SESSION[1])
    r = run_protocol(keys, inp, SESSION, POOL, INPUT_BITS, attack="compensating_collusion", shots=1)
    _check("C10 exact guarantee (accepted output == committed sum)", r["accepted"] and r["S_q"] == tot)
    report["C10"] = "ok"
    # C11
    tr_, o_, _ = base_protocol_trials(3, 3, 20)
    _check("C11 baseline honest", o_ == tr_)
    tr_, o_, dl = base_protocol_trials(3, 3, 20, attack="phase")
    _check("C11 baseline phase attack shifts output by -delta",
           all(o == (t_ - dd) % 8 for o, t_, dd in zip(o_, tr_, dl)))
    tr_, o_, _ = base_protocol_trials(3, 3, 20, attack="broadcast")
    _check("C11 baseline broadcast attack corrupts output", all(o != t_ for o, t_ in zip(o_, tr_)))
    report["C11"] = "ok"
    # C12
    nm = make_noise(p1=0.001, p2=0.01, pro=0.01)
    qc = build_session_circuit(4, [3, 5, 7, 9])
    a1 = run_circuits([qc], 15, nm)[0]
    a2 = run_circuits([qc], 15, nm)[0]
    _check("C12 determinism", a1 == a2 and plurality(a1) == plurality(a2))
    report["C12"] = "ok"
    # C13
    _check("C13 no wrap", all(s_ * ((1 << l_) - 1) < (1 << register_width(s_, l_))
                               for s_ in range(2, 12) for l_ in range(1, 9)))
    report["C13"] = "ok"
    return report

_rng_state, _np_state = RNG.getstate(), NP_RNG.bit_generator.state
consecutive, rnd = 0, 0
t_loop = time.time()
while consecutive < VERIFY_ROUNDS:
    rnd += 1
    if rnd > VERIFY_MAX_ROUNDS:
        raise RuntimeError("verification loop exceeded its round cap")
    RNG.seed(SEED + 7919 * rnd)
    rep = conceptual_round(KEYS, rnd)          # raises AssertionError on the first failing property
    consecutive += 1
    print(f"round {rnd}: all 13 conceptual checks passed | {rep['C8']} | {rep['C9']}")
RNG.setstate(_rng_state)
NP_RNG.bit_generator.state = _np_state
print(f"VERDICT: implementation is conceptually consistent ({consecutive} consecutive clean rounds, "
      f"{time.time() - t_loop:.0f} s).")

## 8. Worked example (printed transcript)

In [ ]:
SESSION = list(range(1, SESSION_SIZE + 1))
POOL = list(range(N_PARTIES, 0, -1))        # decryptors: any k of n, here the last parties first
example_inputs = {j: randbelow(1 << INPUT_BITS) for j in SESSION}
print("n =", N_PARTIES, "k =", THRESHOLD, "session =", SESSION, "inputs =", example_inputs,
      "true sum =", sum(example_inputs.values()))
print("\n--- honest run ---")
tr = run_protocol(KEYS, example_inputs, SESSION, POOL, INPUT_BITS, verbose=True)
print("\n--- input substitution by party", SESSION[1], "---")
tr = run_protocol(KEYS, example_inputs, SESSION, POOL, INPUT_BITS, attack="input_substitution", verbose=True)
print("\n--- false decryption share ---")
tr = run_protocol(KEYS, example_inputs, SESSION, POOL, INPUT_BITS, attack="bad_share", verbose=True)

## E1. Correctness across parameters (noiseless)

In [ ]:
rows = []
for (n_, k_, s_, l_) in [(3, 2, 2, 2), (3, 2, 3, 3), (4, 3, 4, 3), (5, 3, 4, 3), (5, 3, 5, 4), (6, 4, 6, 3)]:
    keys_ = KEYS if (n_, k_) == (N_PARTIES, THRESHOLD) else tp_keygen(KEY_BITS_EXPERIMENTS, n_, k_)
    ok = 0
    trials = max(10, TRIALS // 5)
    for _ in range(trials):
        sess = sorted(RNG.sample(range(1, n_ + 1), s_))
        pool = RNG.sample(range(1, n_ + 1), n_)
        inp = {j: randbelow(1 << l_) for j in sess}
        r = run_protocol(keys_, inp, sess, pool, l_, shots=1)
        ok += int(r["accepted"] and r["S_q"] == r["true_sum"])
    rows.append({"n": n_, "k": k_, "s": s_, "l": l_, "t (qubits)": register_width(s_, l_),
                 "trials": trials, "correct & accepted (%)": 100 * ok / trials})
E1 = savetable(pd.DataFrame(rows), "E1_correctness")
E1

## E2. Noise robustness of the quantum phase
Per-shot success probability and majority-vote success versus the two-qubit error rate p2 (with p1 = p2/10, readout error = p2, channel error = p2/10 per hop), for different register widths.
The acceptance rate of an honest run equals the majority-vote success rate, because verification accepts exactly when S_q equals the committed sum.

In [ ]:
def quantum_only_trials(t, s, eps, trials, shots):
    nm = make_noise(p1=eps / 10, p2=eps, pro=eps, pch=eps / 10)
    circs, sums = [], []
    for _ in range(trials):
        vals = [randbelow(1 << t) for _ in range(s)]
        circs.append(build_session_circuit(t, vals, channel=eps > 0))
        sums.append(sum(vals) % (1 << t))
    outs = run_circuits(circs, shots, nm)
    p_shot = np.mean([o.get(sv, 0) / shots for o, sv in zip(outs, sums)])
    p_major = np.mean([plurality(o) == sv for o, sv in zip(outs, sums)])
    return p_shot, p_major

EPS = [0.0, 0.002, 0.005, 0.01, 0.02, 0.03, 0.05]
rows = []
for t in (3, 4, 5, 6):
    for eps in EPS:
        ps, pm = quantum_only_trials(t, SESSION_SIZE, eps, TRIALS, 101)
        rows.append({"t": t, "p2": eps, "per-shot success": ps, "majority(R=101) success": pm})
E2 = savetable(pd.DataFrame(rows), "E2_noise")
fig, ax = plt.subplots(figsize=(IEEE_COL, 2.4))
for t in (3, 4, 5, 6):
    sub = E2[E2.t == t]
    ax.plot(sub["p2"], sub["per-shot success"], "o-", ms=3, label=f"t={t} per-shot")
    ax.plot(sub["p2"], sub["majority(R=101) success"], "s--", ms=3, label=f"t={t} majority")
ax.set_xlabel("two-qubit depolarizing rate $p_2$")
ax.set_ylabel("success probability")
ax.legend(ncol=4, fontsize=5, loc="upper center", bbox_to_anchor=(0.5, -0.28))
savefig(fig, "E2_noise")

rows = []
for R in (1, 3, 5, 9, 15, 31, 51):
    for eps in (0.01, 0.03):
        _, pm = quantum_only_trials(5, SESSION_SIZE, eps, TRIALS, R)
        rows.append({"R": R, "p2": eps, "majority success": pm})
E2b = savetable(pd.DataFrame(rows), "E2b_repetitions")
fig, ax = plt.subplots(figsize=(IEEE_COL, 2.2))
for eps in (0.01, 0.03):
    sub = E2b[E2b.p2 == eps]
    ax.plot(sub["R"], sub["majority success"], "o-", ms=3, label=f"$p_2$={eps}")
ax.set_xscale("log"); ax.set_xlabel("repetitions R"); ax.set_ylabel("honest acceptance rate (t=5)")
ax.legend()
savefig(fig, "E2b_repetitions")
E2

## E3. Attack detection and identification
The same attacks are run noiseless and under noise (p2 = 0.01, R = SHOTS).
For `none`, the reported "flagged" value is the false-reject rate.
`bb84_eavesdrop`: an intercept-resend attacker on one BB84 link is usually caught by the QBER test. If the sampled QBER happens to pass, the two key copies disagree, the masks no longer sum to zero, and the equality test rejects.

In [ ]:
def attack_campaign(noise_eps):
    nm = make_noise(p1=noise_eps / 10, p2=noise_eps, pro=noise_eps, pch=noise_eps / 10)
    pch = noise_eps / 10
    rows, per_trial = [], []
    for atk in ATTACKS:
        det, ident, false_acc, layers, correct_out = 0, 0, 0, Counter(), 0
        for _ in range(TRIALS):
            inp = {j: randbelow(1 << INPUT_BITS) for j in SESSION}
            tol = 0.125 if noise_eps > 0 else DECOY_ERR_TOL   # noisy runs tolerate 1 decoy error per hop
            r = run_protocol(KEYS, inp, SESSION, POOL, INPUT_BITS, attack=atk, noise=nm, pch=pch, decoy_tol=tol)
            flagged = (not r["accepted"]) or r["detected_by"] is not None
            det += flagged
            layers[r["detected_by"] or "-"] += 1
            if atk in ("bad_share", "out_of_range"):
                ident += (r["identified"] == SESSION[1])
            if atk != "none" and r["accepted"] and r["S_q"] != r["true_sum"]:
                false_acc += 1
            if atk == "bad_share":
                correct_out += (r["accepted"] and r["S_q"] == r["true_sum"])
            per_trial.append({"noise": noise_eps, **{k_: v for k_, v in r.items() if k_ != "decoy_error_rates"}})
        lo, hi = wilson(det, TRIALS)
        row = {"noise p2": noise_eps, "attack": atk, "trials": TRIALS,
               "flagged (%)": 100 * det / TRIALS,
               "95% CI (%)": f"[{100 * lo:.1f}, {100 * hi:.1f}]",
               "layers": dict(layers),
               "false accept (%)": 100 * false_acc / TRIALS}
        if atk == "bad_share":
            row["cheater identified (%)"] = 100 * ident / TRIALS
            row["correct sum still output (%)"] = 100 * correct_out / TRIALS
        elif atk == "out_of_range":
            row["cheater identified (%)"] = 100 * ident / TRIALS
        rows.append(row)
    return rows, per_trial

rows0, pt0 = attack_campaign(0.0)
rows1, pt1 = attack_campaign(0.01)
E3 = savetable(pd.DataFrame(rows0 + rows1), "E3_attacks")
pd.DataFrame(pt0 + pt1).to_csv(os.path.join(OUT, "E3_attacks_per_trial.csv"), index=False)

fig, ax = plt.subplots(figsize=(IEEE_COL, 2.4))
labels = [a for a in ATTACKS]
x = np.arange(len(labels))
v0 = [next(r["flagged (%)"] for r in rows0 if r["attack"] == a) for a in labels]
v1 = [next(r["flagged (%)"] for r in rows1 if r["attack"] == a) for a in labels]
ax.bar(x - 0.2, v0, 0.4, label="noiseless")
ax.bar(x + 0.2, v1, 0.4, label="$p_2$=0.01")
SHORT = {"none": "honest", "input_substitution": "input\nsubst.", "phase_tamper": "phase\ntamper",
         "intercept_resend": "intercept\nresend", "forge_result": "forged\nresult", "bad_share": "bad\nshare",
         "out_of_range": "out of\nrange", "bb84_eavesdrop": "BB84\nEve"}
ax.set_xticks(x); ax.set_xticklabels([SHORT[l] for l in labels], fontsize=5.5)
ax.set_ylabel("flagged / rejected (%)"); ax.set_ylim(0, 120)
ax.legend(fontsize=6, ncol=2, loc="upper center")
ax.annotate("false-reject\nrate", (0, 3), ha="center", fontsize=5)
savefig(fig, "E3_attacks")
E3

## E4. Decoy-qubit detection of intercept-resend (stabilizer simulation vs theory 1-(3/4)^D)

In [ ]:
rows = []
for D in range(1, 13):
    flagged = 0
    reps = 20 * TRIALS
    rates = run_decoy_checks([True] * reps, D)
    flagged = sum(r > 0 for r in rates)
    lo, hi = wilson(flagged, reps)
    rows.append({"D": D, "empirical": flagged / reps, "ci_low": lo, "ci_high": hi,
                 "theory": 1 - 0.75 ** D, "theory inside CI": lo <= 1 - 0.75 ** D <= hi, "trials": reps})
E4 = savetable(pd.DataFrame(rows), "E4_decoys")
fig, ax = plt.subplots(figsize=(IEEE_COL, 2.2))
ax.plot(E4.D, E4.theory, "-", label="theory $1-(3/4)^D$")
ax.errorbar(E4.D, E4.empirical, yerr=[E4.empirical - E4.ci_low, E4.ci_high - E4.empirical],
            fmt="o", ms=3, capsize=2, label="simulation (95% CI)")
ax.set_xlabel("decoys per hop D"); ax.set_ylabel("detection probability"); ax.legend()
savefig(fig, "E4_decoys")
E4

## E5. Classical cost versus key size
Times are per operation (mean over BENCH_REPS runs), measured in this runtime with gmpy2.

In [ ]:
rows = []
for bits in KEY_BITS_BENCH:
    t0 = time.time(); ks = tp_keygen(bits, N_PARTIES, THRESHOLD); tk = time.time() - t0
    pk = ks.pk
    tc, tv, ts, tsv, tcomb = [], [], [], [], []
    for rep in range(BENCH_REPS):
        val = randbelow(1 << INPUT_BITS)
        t0 = time.time(); C, cts, prf = commit_input(pk, val, INPUT_BITS, 1, rep); tc.append(time.time() - t0)
        t0 = time.time(); assert verify_commitment(pk, C, cts, prf, 1, rep); tv.append(time.time() - t0)
        sh = {}
        for j in range(1, THRESHOLD + 1):
            t0 = time.time(); cj, pr = decryption_share(ks, j, C); ts.append(time.time() - t0)
            t0 = time.time(); assert verify_share(pk, j, C, cj, pr); tsv.append(time.time() - t0)
            sh[j] = cj
        t0 = time.time(); assert combine_shares(pk, sh) == val; tcomb.append(time.time() - t0)
    rows.append({"N bits": bits, "keygen (s)": tk,
                 f"commit+range proof, l={INPUT_BITS} (ms)": 1e3 * np.mean(tc),
                 "verify commitment (ms)": 1e3 * np.mean(tv),
                 "decryption share+proof (ms)": 1e3 * np.mean(ts),
                 "verify share (ms)": 1e3 * np.mean(tsv),
                 f"combine k={THRESHOLD} (ms)": 1e3 * np.mean(tcomb),
                 "ciphertext size (bits)": 2 * bits})
E5 = savetable(pd.DataFrame(rows), "E5_classical_cost")
fig, ax = plt.subplots(figsize=(IEEE_COL, 2.4))
cols = [c for c in E5.columns if c.endswith("(ms)")]
xx = np.arange(len(E5))
w = 0.8 / len(cols)
for i, ccol in enumerate(cols):
    ax.bar(xx + i * w - 0.4 + w / 2, E5[ccol], w, label=ccol.replace(" (ms)", ""))
ax.set_yscale("log"); ax.set_xticks(xx); ax.set_xticklabels(E5["N bits"])
ax.set_xlabel("Paillier modulus size (bits)"); ax.set_ylabel("time per operation (ms)")
ax.legend(fontsize=5)
savefig(fig, "E5_classical_cost")
E5

## E6. Quantum resources
Logical gate counts per session, plus CX counts of the only entangling blocks (P1's QFT and the last party's QFT^-1), each transpiled separately to the IBM basis {rz, sx, x, cx} (optimization level 3, all-to-all connectivity).
Middle parties apply only single-qubit phase gates. Blocks belonging to different parties are never merged, because they run in different laboratories.

In [ ]:
BASIS = ["rz", "sx", "x", "cx"]
rows = []
for t in range(2, 9):
    q_f = transpile(qft_circ(t), basis_gates=BASIS, optimization_level=3, seed_transpiler=SEED)
    q_i = transpile(qft_circ(t).inverse(), basis_gates=BASIS, optimization_level=3, seed_transpiler=SEED)
    cx_f, cx_i = q_f.count_ops().get("cx", 0), q_i.count_ops().get("cx", 0)
    for s in (3, 5, 8):
        qc = build_session_circuit(t, [randbelow(1 << t) for _ in range(s)])
        ops = qc.count_ops()
        rows.append({"t": t, "s": s, "signal qubits": t,
                     "qubits sent (incl. decoys)": (s - 1) * (t + DECOYS_PER_HOP),
                     "H": ops.get("h", 0), "CP": ops.get("cp", 0), "SWAP": ops.get("swap", 0),
                     "P (phase adds, 1q)": ops.get("p", 0), "measurements (signal)": ops.get("measure", 0),
                     "CX in QFT (P1)": cx_f, "CX in QFT^-1 (last party)": cx_i,
                     "total CX": cx_f + cx_i, "CX per middle party": 0,
                     "QFT depth": q_f.depth(), "QFT^-1 depth": q_i.depth()})
E6 = savetable(pd.DataFrame(rows), "E6_quantum_resources")
fig, ax = plt.subplots(figsize=(IEEE_COL, 2.2))
sub = E6[E6.s == 5]
ax.plot(sub.t, sub["total CX"], "o-", ms=3, label="total CX (QFT + QFT$^{-1}$)")
ax.plot(sub.t, sub["P (phase adds, 1q)"], "s-", ms=3, label="1-qubit phase gates (s=5)")
ax.plot(sub.t, sub["QFT depth"], "^-", ms=3, label="QFT depth")
ax.set_xlabel("register width t"); ax.set_ylabel("count"); ax.legend(fontsize=6)
savefig(fig, "E6_quantum_resources")
E6

## E7. IBM device noise (fake backends, optional)
Transpiles to a real device's coupling map and basis, then simulates it with that device's calibrated noise.

In [ ]:
E7 = None
if RUN_FAKE_BACKEND:
    try:
        import qiskit_ibm_runtime.fake_provider as fp
        rows = []
        for fname in ("FakeBrisbane", "FakeSherbrooke", "FakeTorino"):
            try:
                dev = getattr(fp, fname)()
            except Exception as ex:
                print("skip", fname, ex); continue
            sim = AerSimulator.from_backend(dev, seed_simulator=SEED)
            for t in (2, 3, 4, 5):
                circs, sums = [], []
                for _ in range(max(10, TRIALS // 5)):
                    vals = [randbelow(1 << t) for _ in range(SESSION_SIZE)]
                    circs.append(build_session_circuit(t, vals))
                    sums.append(sum(vals) % (1 << t))
                tq = transpile(circs, backend=dev, optimization_level=3, seed_transpiler=SEED)
                res = sim.run(tq, shots=1024).result()
                ps, pm = [], []
                for i, sv in enumerate(sums):
                    c = Counter()
                    for key, v in res.get_counts(i).items():
                        c[int(key.split()[-1], 2)] += v
                    ps.append(c.get(sv, 0) / 1024)
                    pm.append(plurality(c) == sv)
                rows.append({"device": dev.name, "t": t, "per-shot success": float(np.mean(ps)),
                             "majority(1024) success": float(np.mean(pm)),
                             "mean transpiled 2q gates": float(np.mean([sum(v for k_, v in q_.count_ops().items() if k_ in ("cx", "ecr", "cz")) for q_ in tq]))})
        E7 = savetable(pd.DataFrame(rows), "E7_fake_backends")
        fig, ax = plt.subplots(figsize=(IEEE_COL, 2.2))
        for dname in E7.device.unique():
            sub = E7[E7.device == dname]
            ax.plot(sub.t, sub["per-shot success"], "o-", ms=3, label=dname)
        ax.plot(E7.t.unique(), [1 / (1 << t) for t in E7.t.unique()], "k:", label="random guess")
        ax.set_xticks(sorted(E7.t.unique()))
        ax.set_xlabel("register width t"); ax.set_ylabel("per-shot success"); ax.legend(fontsize=6)
        savefig(fig, "E7_fake_backends")
    except ImportError as ex:
        print("qiskit-ibm-runtime not available:", ex)
E7

## E9. Head-to-head with the base protocol (Sutradhar & Om, TCAS-II 2020)
The same two attacks are applied to the qubit emulation of the base protocol and to the proposed protocol, noiseless, with TRIALS runs each:
* **phase shift** on one travelling register (invisible to measurements of that register);
* **false broadcast / forged result** by one dishonest participant.

"Silent corruption" means the protocol output a wrong sum and raised no alarm.
"Correct behaviour" means accepting the true sum in honest runs and rejecting under attack.

In [ ]:
rows = []
for atk, label in (("none", "honest"), ("phase", "phase shift on channel"), ("broadcast", "false broadcast / forged result")):
    tr_, o_, _ = base_protocol_trials(3, SESSION_SIZE, TRIALS, attack=atk)
    wrong = sum(o != t_ for o, t_ in zip(o_, tr_))
    rows.append({"protocol": "Sutradhar & Om [19] (emulated)", "attack": label, "trials": TRIALS,
                 "correct sum output (%)": 100 * (TRIALS - wrong) / TRIALS, "attack detected (%)": 0.0,
                 "silent corruption (%)": 100 * wrong / TRIALS,
                 "correct behaviour (%)": 100 * (TRIALS - wrong) / TRIALS if atk == "none" else 0.0})
for atk, label in (("none", "honest"), ("phase_tamper", "phase shift on channel"),
                   ("forge_result", "false broadcast / forged result")):
    det = silent = ok = 0
    for _ in range(TRIALS):
        inp = {j: randbelow(1 << INPUT_BITS) for j in SESSION}
        r = run_protocol(KEYS, inp, SESSION, POOL, INPUT_BITS, attack=atk, shots=1)
        correct = r["accepted"] and r["S_q"] == r["true_sum"]
        ok += correct
        det += (not r["accepted"])
        silent += (r["accepted"] and r["S_q"] != r["true_sum"])
    rows.append({"protocol": "Proposed", "attack": label, "trials": TRIALS,
                 "correct sum output (%)": 100 * ok / TRIALS,
                 "attack detected (%)": 100 * det / TRIALS if atk != "none" else 0.0,
                 "silent corruption (%)": 100 * silent / TRIALS,
                 "correct behaviour (%)": 100 * ok / TRIALS if atk == "none" else 100 * det / TRIALS})
E9 = savetable(pd.DataFrame(rows), "E9_head_to_head")
fig, ax = plt.subplots(figsize=(IEEE_COL, 2.3))
labs = ["honest", "phase shift on channel", "false broadcast / forged result"]
xx = np.arange(len(labs))
b_sc = [E9[(E9.protocol.str.startswith("Sutradhar")) & (E9.attack == l)]["silent corruption (%)"].iloc[0] for l in labs]
p_sc = [E9[(E9.protocol == "Proposed") & (E9.attack == l)]["silent corruption (%)"].iloc[0] for l in labs]
p_det = [E9[(E9.protocol == "Proposed") & (E9.attack == l)]["attack detected (%)"].iloc[0] for l in labs]
ax.bar(xx - 0.27, b_sc, 0.27, label="[19]: silent corruption")
ax.bar(xx, p_sc, 0.27, label="proposed: silent corruption")
ax.bar(xx + 0.27, p_det, 0.27, label="proposed: detected")
ax.set_xticks(xx); ax.set_xticklabels(["honest", "phase shift", "false broadcast"], fontsize=6)
ax.set_ylabel("% of trials"); ax.set_ylim(0, 135)
ax.legend(fontsize=5, loc="upper center", ncol=3, columnspacing=0.8, handlelength=1.2)
savefig(fig, "E9_head_to_head")
E9

## E8. IBM quantum hardware (optional)
Runs the quantum phase of complete sessions on a device, then performs the classical verification locally:
honest sessions and phase-tampered sessions for t = 2 and t = 3.
* `E8_DRY_RUN = True` runs exactly the same code on a local fake backend. This needs no account, uses no QPU minutes, and lets you test the cell first.
* `RUN_ON_IBM_HARDWARE = True` (with IBM_TOKEN, and ideally IBM_INSTANCE) submits one job of 4·HW_TRIALS circuits × 1024 shots.
* Decoy checks and BB84 remain simulated; a single chip cannot emulate separate laboratories or a real quantum channel. State this in the paper.

In [ ]:
E8 = None
if RUN_ON_IBM_HARDWARE or E8_DRY_RUN:
    from qiskit_ibm_runtime import SamplerV2 as Sampler
    from qiskit.transpiler import generate_preset_pass_manager
    if RUN_ON_IBM_HARDWARE:
        from qiskit_ibm_runtime import QiskitRuntimeService
        service = QiskitRuntimeService(channel="ibm_quantum_platform", token=IBM_TOKEN, instance=IBM_INSTANCE)
        backend = service.least_busy(operational=True, simulator=False)
    else:
        import qiskit_ibm_runtime.fake_provider as fp
        backend = fp.FakeTorino()
    pm = generate_preset_pass_manager(backend=backend, optimization_level=3, seed_transpiler=SEED)
    jobs_meta, circs = [], []
    for s_hw in (3, 4):                           # t = register_width(s, 1) -> 2 and 3
        ell_hw = 1
        t_hw = register_width(s_hw, ell_hw)
        d_hw = 1 << t_hw
        sess_hw = list(range(1, s_hw + 1))
        for atk in ("none", "phase_tamper"):
            for _ in range(HW_TRIALS):
                sid = randbits(64)
                inp = {j: randbelow(1 << ell_hw) for j in sess_hw}
                commits = {}
                for j in sess_hw:
                    C, cts, prf = commit_input(PK, inp[j], ell_hw, j, sid)
                    assert verify_commitment(PK, C, cts, prf, j, sid)
                    commits[j] = C
                masks, _, _ = zero_sum_masks(sess_hw, t_hw)
                contrib = [(inp[j] + masks[j]) % d_hw for j in sess_hw]
                hop = {randbelow(s_hw - 1): ("phase", 1 + randbelow(d_hw - 1))} if atk == "phase_tamper" else {}
                circs.append(build_session_circuit(t_hw, contrib, hop))
                jobs_meta.append({"t": t_hw, "attack": atk, "sid": sid, "session": sess_hw,
                                  "commits": commits, "true_sum": sum(inp.values())})
    if RUN_ON_IBM_HARDWARE:
        sampler = Sampler(mode=backend)                  # hardware shots cannot be seeded
    else:
        sampler = Sampler(mode=backend, options={"simulator": {"seed_simulator": SEED}})
    job = sampler.run(pm.run(circs), shots=1024)
    print("backend:", backend.name, "| job id:", job.job_id())
    result = job.result()
    rows = []
    for meta, pub in zip(jobs_meta, result):
        counts = Counter({int(k_, 2): v for k_, v in pub.data.out.get_counts().items()})
        S_q = plurality(counts)
        if S_q is None:
            accepted = False
        else:
            accepted = equality_test(KEYS, meta["commits"], S_q, meta["session"], POOL, meta["sid"])["accepted"]
        rows.append({"backend": backend.name, "t": meta["t"], "attack": meta["attack"],
                     "true sum": meta["true_sum"], "S_q (plurality)": S_q,
                     "per-shot success": counts.get(meta["true_sum"], 0) / 1024,
                     "accepted": accepted,
                     "outcome correct": (accepted and S_q == meta["true_sum"]) if meta["attack"] == "none"
                                        else (not accepted)})
    E8 = savetable(pd.DataFrame(rows), "E8_ibm_hardware" if RUN_ON_IBM_HARDWARE else "E8_dry_run_fake_backend")
    print(E8.groupby(["t", "attack"])[["per-shot success", "outcome correct"]].mean())
E8

## Export

In [ ]:
meta = {"seed": SEED, "n": N_PARTIES, "k": THRESHOLD, "s": SESSION_SIZE, "l": INPUT_BITS,
        "key_bits_experiments": KEY_BITS_EXPERIMENTS, "decoys_per_hop": DECOYS_PER_HOP,
        "trials": TRIALS, "shots": SHOTS}
import platform
meta.update(python=platform.python_version(), machine=platform.machine(),
            processor=platform.processor() or "n/a", cpu_count=os.cpu_count())
try:
    import qiskit as _q, qiskit_aer as _qa
    meta.update(qiskit=_q.__version__, qiskit_aer=_qa.__version__, gmpy2=gmpy2.version())
except Exception:
    pass
with open(os.path.join(OUT, "run_metadata.json"), "w") as f:
    json.dump(meta, f, indent=2)
zip_name = "results.zip"
with zipfile.ZipFile(zip_name, "w") as z:
    for fn in sorted(os.listdir(OUT)):
        z.write(os.path.join(OUT, fn), fn)
print("Saved:", sorted(os.listdir(OUT)))
try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    print("Not in Colab; results are in", os.path.abspath(OUT))

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
import numpy as np
service = QiskitRuntimeService(channel="ibm_quantum_platform",
                               token=IBM_TOKEN, instance=IBM_INSTANCE)
job = service.job("dalc3r5r85ps73fb8pig")
print(job.creation_date, job.metrics())          # usage -> quantum_seconds
props = service.backend("ibm_marrakesh").properties()
print(props.last_update_date,
      np.median([g.parameters[0].value for g in props.gates
                 if g.gate in ("cz", "ecr") and g.parameters]))